# Kakunin + OpenAI Assistants API Compliance Playground

This notebook demonstrates how to build and execute compliance-guarded AI agents using the stateful OpenAI Assistants API and Kakunin.

### Step 1: Install Dependencies
First, we'll install `kakunin`, `openai`, `python-dotenv`.

In [ ]:
!pip install kakunin openai python-dotenv

### Step 2: Configure Environment Keys
Set your Kakunin and OpenAI API keys below.

In [ ]:
import os
import getpass

os.environ["KAK_API_KEY"] = getpass.getpass("Enter Kakunin API Key (kak_live_...): ")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key (sk-...): ")

### Step 3: Run the Assistants API Demo
Now we'll run a script to register the agent with Kakunin, issue its compliance certificate, bind tool scopes, and run the agent.

In [ ]:
import asyncio
from openai import OpenAI
from kakunin import Kakunin
from kakunin.exceptions import ScopeViolationError
from kakunin.integrations.openai_assistants import handle_assistants_requires_action

# Define mock tools
def query_market_prices(symbol: str) -> str:
    """Query current prices for a given ticker symbol."""
    print(f"[Tool Executing] query_market_prices: {symbol}")
    return f"Latest price for {symbol}: $150.00"

def execute_market_trade(symbol: str, amount: int) -> str:
    """Execute a market buy order for a symbol."""
    print(f"[Tool Executing] execute_market_trade: Buying {amount} of {symbol}")
    return f"Successfully executed trade: Buy {amount} shares of {symbol}"

async def run_compliance_demo():
    openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    
    async with Kakunin(api_key=os.environ["KAK_API_KEY"]) as kakunin_client:
        # 1. Register agent
        agent_record = await kakunin_client.agents.create(
            name="AssistantsTrader",
            model="gpt-4o",
            version="1.0.0",
            model_hash="sha256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
        )
        print(f"Registered Agent: {agent_record.id}")

        # 2. Issue certificate
        cert = await kakunin_client.agents.certify(agent_record.id)
        print(f"Issued Certificate Serial: {cert.serial_number}")

        # 3. Create OpenAI Assistant
        assistant = openai_client.beta.assistants.create(
            name="Compliance Trader",
            instructions="You are a stock trading assistant. You can check prices and buy shares using tools.",
            model="gpt-4o",
            tools=[
                {
                    "type": "function",
                    "function": {
                        "name": "query_market_prices",
                        "description": "Query current prices for a symbol.",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "symbol": {"type": "string"}
                            },
                            "required": ["symbol"]
                        }
                    }
                },
                {
                    "type": "function",
                    "function": {
                        "name": "execute_market_trade",
                        "description": "Execute a market buy order.",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "symbol": {"type": "string"},
                                "amount": {"type": "integer"}
                            },
                            "required": ["symbol", "amount"]
                        }
                    }
                }
            ]
        )

        # 4. Initialize Thread
        thread = openai_client.beta.threads.create()

        tool_scopes_mapping = {
            "query_market_prices": ["market.read"],
            "execute_market_trade": ["trade.execute"]
        }
        tool_funcs = {
            "query_market_prices": query_market_prices,
            "execute_market_trade": execute_market_trade
        }

        openai_client.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content="Check price of GOOG and then buy 5 shares of GOOG"
        )

        run = openai_client.beta.threads.runs.create(
            thread_id=thread.id,
            assistant_id=assistant.id
        )

        while True:
            run = openai_client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
            print(f"Run Status: {run.status}")

            if run.status == "requires_action":
                print("Enforcing Kakunin compliance check...")
                tool_outputs = await handle_assistants_requires_action(
                    kakunin=kakunin_client,
                    agent_id=agent_record.id,
                    run=run,
                    tool_scopes_mapping=tool_scopes_mapping,
                    tool_funcs=tool_funcs,
                    catch_exceptions=True
                )
                openai_client.beta.threads.runs.submit_tool_outputs(
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs
                )
            elif run.status == "completed":
                messages = openai_client.beta.threads.messages.list(thread_id=thread.id)
                print(f"\nResponse:\n{messages.data[0].content[0].text.value}")
                break
            elif run.status in ["failed", "cancelled"]: 
                print("Execution failed")
                break
            await asyncio.sleep(1)
            
        # Cleanup
        openai_client.beta.assistants.delete(assistant.id)

await run_compliance_demo()